In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Definició del directori arrel del projecte
ROOT = Path.cwd()
ROOT = ROOT.parents[1]

ruta_csv = ROOT / "data" / "processed" / "education" / "resultats_academics_quart_eso_processat_nou.csv"

# Llegim el CSV gestionant el separador ';' i possibles problemes de codificació d'accents
df = pd.read_csv(ruta_csv, sep=";", encoding="utf-8", encoding_errors="replace")
if df.shape[1] == 1:
    df = pd.read_csv(ruta_csv, sep=",", encoding="utf-8", encoding_errors="replace")

# Construïm un DataFrame net accedint per índex de columna (0 a 7)
df_net = pd.DataFrame({
    "ANY": pd.to_numeric(df.iloc[:, 0], errors="coerce"),
    "PCAT": pd.to_numeric(df.iloc[:, 1].astype(str).str.replace(",", ".", regex=False), errors="coerce"),
    "PCAT_CL": pd.to_numeric(df.iloc[:, 2].astype(str).str.replace(",", ".", regex=False), errors="coerce"),
    "PCAT_EE": pd.to_numeric(df.iloc[:, 3].astype(str).str.replace(",", ".", regex=False), errors="coerce"),
    "PMAT": pd.to_numeric(df.iloc[:, 4].astype(str).str.replace(",", ".", regex=False), errors="coerce"),
    "NATURALESA": df.iloc[:, 5].astype(str).str.strip(),
    "AREA_TERRITORIAL": df.iloc[:, 6].astype(str).str.strip(),
    "HABITAT": df.iloc[:, 7].astype(str).str.strip()
})

# Filtre 1: Escoles públiques (columna F sense "Privada" i no buida)
cond_publica = (
    (~df_net["NATURALESA"].str.contains("privada", case=False, na=False))
    & (df_net["NATURALESA"] != "")
    & (df_net["NATURALESA"].str.lower() != "nan")
)

# Filtre 2: Municipis de fins a 100.000 habitants (columna H)
cond_habitat = df_net["HABITAT"].isin(["Fins a 10000", "De 10001 a 100000"])

df_filtrat = df_net[cond_publica & cond_habitat].dropna(subset=["ANY"]).copy()
df_filtrat["ANY"] = df_filtrat["ANY"].astype(int)

# Calculem la mitjana anual per a les 4 competències (sense distinció de sexe)
df_mitjanes = (
    df_filtrat
    .groupby("ANY", as_index=False)[["PCAT", "PCAT_CL", "PCAT_EE", "PMAT"]]
    .mean()
    .sort_values("ANY")
)

print(f"Files totals després del filtre: {len(df_filtrat)}")
display(df_mitjanes)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\carle\\Desktop\\Master\\codi\\data\\processed\\education\\resultats_academics_quart_eso_processat_nou.csv'

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=150)

competencies = [
    ("PCAT", "Catalan (PCAT)", axes[0, 0]),
    ("PCAT_CL", "Catalan Reading Comprehension (PCAT_CL)", axes[0, 1]),
    ("PCAT_EE", "Catalan Written Skills (PCAT_EE)", axes[1, 0]),
    ("PMAT", "Mathematics (PMAT)", axes[1, 1])
]

for col, titol, ax in competencies:
    ax.plot(
        df_mitjanes["ANY"],
        df_mitjanes[col],
        color="royalblue",
        marker="o",
        markersize=5,
        linewidth=1.8,
        label="Public schools (≤ 100,000 inh.)"
    )
    
    ax.set_title(titol, fontsize=11, fontweight="bold", pad=8)
    ax.set_xlabel("Academic year", fontsize=9.5)
    ax.set_ylabel("Mean grade", fontsize=9.5)
    ax.set_ylim(50, 90)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.tick_params(axis="both", labelsize=9)
    ax.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9, fontsize=8.5)

fig.suptitle(
    "4th of ESO evaluation evolution (Public schools, ≤ 100,000 inhabitants)",
    fontsize=13,
    fontweight="bold",
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.95])

# Desem la figura a la carpeta de resultats del repositori
ruta_figura = ROOT / "results" / "figures" / "evolucio_4t_eso_publica_100k.png"
ruta_figura.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(ruta_figura, dpi=300, bbox_inches="tight")

plt.show()